# Planemo Tests

### 1. Run Knime Workflow to collect output

Run in CLI:

`knime \
  -nosplash \
  -application org.knime.product.KNIME_BATCH_APPLICATION \
  -workflowFile="../data/file_to_translate/2025_03_2D_spot_detection.knwf" \
  -workflow.variable=input_file,"../knime2galaxy_scheme.png",String \
  -workflow.variable=output_dir,"../data/knime_out",String \
  -reset \
  -nosave
`

In [ ]:
!knime \
  -nosplash \
  -application org.knime.product.KNIME_BATCH_APPLICATION \
  -workflowFile="../data/file_to_translate/2025_03_2D_spot_detection_gui.knwf" \
  -workflow.variable=input_file,"../knime2galaxy_scheme.png",String \
  -workflow.variable=output_dir,"../data/test.csv",String \
  -reset \
  -nosave

## 2. Create equivalent .ga output

In [ ]:
from imaging_knime_to_galaxy.translate import translate_knime_to_galaxy
from pathlib import Path

data_folder = Path("../data").resolve()

workflow = translate_knime_to_galaxy(
    knwf_path=data_folder / "file_to_translate" / "2025_03_2D_spot_detection_gui.knwf",
    tools_metadata_path=data_folder / "tools_metadata.json",
    translation_table_path=data_folder / "translation_table.yml",
    workflow_examples_yml_path=data_folder / "workflow_translation_table.yml",
    output_galaxy_workflow_path=data_folder / "output_file" / "gui_test_file.ga",
    input_workflow_path =data_folder / "input_workflows.ga",
    vector_store_path =data_folder/ "vector_store.npz",
    
)

## 3. Use planemo to test validity of the new .ga file

In [ ]:
import subprocess
import os

results = []
outputs = os.listdir(data_folder / "output_file")
valid_results = 0

for output in outputs:
    # find desired .ga file
    if output.startswith("gui_test_file.ga"):    
        # run planemo's workflow_lint function and use subprocess, as planemo is a command line tool
        result = subprocess.run(
            ["planemo", "workflow_lint", data_folder / "output_file" / f"{output}"],
            capture_output=True,
            text=True
        )
        stdout = result.stdout
        stderr = result.stderr
        has_error = "ERROR" in stdout or "ERROR" in stderr # check whether the result holds an error
    
        if has_error:
            print("Lint errors present for workflow", output)
            print(stdout)
        else:
            print("Lint passed for workflow", output)
            valid_results += 1
    
        results.append(result)
    else:
        continue

## 4. Run the .ga file with the same input as the .knwf file

### generate a labeled version of the .ga file to create a valid job .yml based on the labels:

In [ ]:
import json
from pathlib import Path

INPUT_STEP_TYPES = {"data_input", "data_collection_input", "parameter_input"}

def add_missing_input_labels(ga_path: str | Path, output_path: str | Path | None = None):
    ga_path = Path(ga_path)
    if output_path is None:
        output_path = ga_path
    else:
        output_path = Path(output_path)

    with ga_path.open("r", encoding="utf-8") as f:
        workflow = json.load(f)

    steps = workflow.get("steps", {})
    changed = []

    for step_id, step in steps.items():
        if step.get("type") in INPUT_STEP_TYPES and not step.get("label"):
            label = f"input_{step_id}"
            step["label"] = label
            changed.append((step_id, label))

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(workflow, f, indent=2)

    print(f"Patched workflow written to {output_path}")
    print("Added labels:")
    for step_id, label in changed:
        print(f"  step {step_id} -> {label}")

In [ ]:
add_missing_input_labels("/imaging_KNIME_to_Galaxy/data/output_file/gui_test_file.ga", "/imaging_KNIME_to_Galaxy/data/output_file/gui_test_file_labeled.ga")

### for this to work we also need to generate a .yml job file:

In [ ]:
import json
from pathlib import Path
import yaml


INPUT_STEP_TYPES = {"data_input", "data_collection_input", "parameter_input"}

def generate_job_yaml(ga_path, output_path, default_file):
    ga_path = Path(ga_path)
    output_path = Path(output_path)
    default_file = str(Path(default_file).resolve())

    with ga_path.open("r", encoding="utf-8") as f:
        workflow = json.load(f)

    job = {}
    for step_id, step in workflow.get("steps", {}).items():
        if step.get("type") in INPUT_STEP_TYPES:
            key = step.get("label") or f"input_{step_id}"
            job[key] = {
                "class": "File",
                "path": default_file,
            }

    with output_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(job, f, sort_keys=False)

    print(yaml.safe_dump(job, sort_keys=False))

In [ ]:
# generate the file
generate_job_yaml(
    "/imaging_KNIME_to_Galaxy/data/output_file/gui_test_file_labeled.ga",
    "/imaging_KNIME_to_Galaxy/data/job.yml",
    "/imaging_KNIME_to_Galaxy/knime2galaxy_scheme.png"
)

In [ ]:
!planemo run /imaging_KNIME_to_Galaxy/data/output_file/gui_test_file_labeled.ga /imaging_KNIME_to_Galaxy/data/job.yml --engine external_galaxy \
  --galaxy_url https://usegalaxy.eu \
  --galaxy_user_key $GALAXY_API_KEY

## 5. Load the .ga file and analysze the single steps

In [ ]:
import json
from pathlib import Path
from typing import Any


INPUT_STEP_TYPES = {"data_input", "data_collection_input", "parameter_input"}


def load_ga_workflow(path: str | Path) -> dict[str, Any]:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def summarize_steps(workflow: dict[str, Any]) -> list[dict[str, Any]]:
    steps = workflow.get("steps", {})
    summary = []

    for raw_step_id, step in steps.items():
        step_id = int(raw_step_id) if str(raw_step_id).isdigit() else raw_step_id
        step_type = step.get("type")
        name = step.get("name")
        label = step.get("label")
        tool_id = step.get("tool_id")
        content_id = step.get("content_id")
        tool_version = step.get("tool_version")
        input_connections = step.get("input_connections", {})
        outputs = step.get("outputs", [])
        errors = step.get("errors")

        summary.append(
            {
                "step_id": step_id,
                "type": step_type,
                "name": name,
                "label": label,
                "tool_id": tool_id,
                "content_id": content_id,
                "tool_version": tool_version,
                "num_inputs": len(input_connections),
                "num_outputs": len(outputs),
                "errors": errors,
            }
        )

    summary.sort(key=lambda x: (isinstance(x["step_id"], str), x["step_id"]))
    return summary


def _parse_tool_state(tool_state: Any) -> dict[str, Any]:
    if isinstance(tool_state, dict):
        return tool_state
    if isinstance(tool_state, str):
        try:
            return json.loads(tool_state)
        except json.JSONDecodeError:
            return {}
    return {}


def analyze_workflow_inputs(workflow: dict[str, Any]) -> list[dict[str, Any]]:
    """Find workflow-level inputs and whether they appear optional."""
    steps = workflow.get("steps", {})
    inputs = []

    for raw_step_id, step in steps.items():
        step_type = step.get("type")
        if step_type not in INPUT_STEP_TYPES:
            continue

        step_id = int(raw_step_id) if str(raw_step_id).isdigit() else raw_step_id
        tool_state = _parse_tool_state(step.get("tool_state", {}))

        # For many Galaxy exports, optional is stored in tool_state
        optional = tool_state.get("optional")
        if optional is None:
            optional = False

        row = {
            "step_id": step_id,
            "type": step_type,
            "name": step.get("name"),
            "label": step.get("label"),
            "optional": bool(optional),
            "tool_state": tool_state,
        }
        inputs.append(row)

    inputs.sort(key=lambda x: (isinstance(x["step_id"], str), x["step_id"]))
    return inputs


def analyze_workflow_steps(workflow: dict[str, Any]) -> dict[str, Any]:
    steps = workflow.get("steps", {})
    problems: list[str] = []
    tool_steps_missing_ids: list[dict[str, Any]] = []
    input_like_steps: list[dict[str, Any]] = []
    valid_tool_steps: list[dict[str, Any]] = []

    for raw_step_id, step in steps.items():
        step_id = int(raw_step_id) if str(raw_step_id).isdigit() else raw_step_id
        step_type = step.get("type")
        name = step.get("name")
        label = step.get("label")
        tool_id = step.get("tool_id")
        content_id = step.get("content_id")

        row = {
            "step_id": step_id,
            "type": step_type,
            "name": name,
            "label": label,
            "tool_id": tool_id,
            "content_id": content_id,
        }

        if step_type in INPUT_STEP_TYPES:
            input_like_steps.append(row)
            if not label:
                problems.append(
                    f"Input step {step_id} ('{name}') has no label. "
                    "You may need to map it by step id when invoking."
                )
        elif step_type == "tool":
            if not tool_id and not content_id:
                tool_steps_missing_ids.append(row)
                problems.append(
                    f"Tool step {step_id} ('{name}') is missing both tool_id and content_id."
                )
            else:
                valid_tool_steps.append(row)

    return {
        "total_steps": len(steps),
        "input_like_steps": input_like_steps,
        "valid_tool_steps": valid_tool_steps,
        "tool_steps_missing_ids": tool_steps_missing_ids,
        "problems": problems,
    }


def build_bioblend_input_template(workflow: dict[str, Any], dataset_placeholder: str = "DATASET_ID") -> dict[str, Any]:
    """
    Build a template dict for gi.workflows.invoke_workflow(..., inputs=...).
    Use label if present, otherwise use step_id as string.
    """
    template = {}
    for wf_input in analyze_workflow_inputs(workflow):
        key = wf_input["label"] if wf_input["label"] else str(wf_input["step_id"])
        template[key] = {"src": "hda", "id": dataset_placeholder}
    return template


def print_step_report(workflow: dict[str, Any]) -> None:
    workflow_name = workflow.get("name", "<unnamed workflow>")
    analysis = analyze_workflow_steps(workflow)
    summary = summarize_steps(workflow)
    wf_inputs = analyze_workflow_inputs(workflow)

    print(f"Workflow: {workflow_name}")
    print(f"Total steps: {analysis['total_steps']}")
    print()

    print("Step summary:")
    for row in summary:
        print(
            f"  Step {row['step_id']:>3} | "
            f"type={row['type']!r:<22} | "
            f"name={str(row['name'])!r:<30} | "
            f"label={str(row['label'])!r:<20} | "
            f"tool_id={str(row['tool_id'])!r}"
        )

    print()
    print("Workflow inputs:")
    if not wf_inputs:
        print("  No workflow inputs found.")
    else:
        for row in wf_inputs:
            print(
                f"  Step {row['step_id']:>3} | "
                f"type={row['type']!r:<22} | "
                f"name={str(row['name'])!r:<30} | "
                f"label={str(row['label'])!r:<20} | "
                f"optional={row['optional']}"
            )

    print()
    if analysis["problems"]:
        print("Problems found:")
        for problem in analysis["problems"]:
            print(f"  - {problem}")
    else:
        print("No missing tool identifiers found for tool steps.")

    print()
    print("BioBlend input template:")
    print(json.dumps(build_bioblend_input_template(workflow), indent=2))

In [ ]:
ga_path = data_folder / "output_file" / "gui_test_file_labeled.ga"

workflow = load_ga_workflow(ga_path)
print_step_report(workflow)

## The above steps are summarized in one [Script](../src/testing_pipeline.py)
### =========== Results of the `testing_pipeline.py` Script ===========
#### 1. Load the `.csv` files

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os

data_folder = Path("../data").resolve()

# load files
df_results = pd.read_csv(os.path.join(data_folder,"pipeline_run_results.csv"))
error_counts = pd.read_csv(os.path.join(data_folder,"error_counts.csv"))
stage_error_counts = pd.read_csv(os.path.join(data_folder,"stage_error_counts.csv"))
file_error_counts = pd.read_csv(os.path.join(data_folder,"file_error_counts.csv"))

#### 2. Visualize overall success vs failure of the pipeline

In [ ]:
df_results["status"].value_counts().plot(kind="bar")
plt.ylabel("Count")
plt.title("Overall pipeline results")
plt.xticks(rotation=0)
plt.show()

#### 2. Visualize errors by stage

In [ ]:
pivot_stage = stage_error_counts.pivot(
    index="failed_stage",
    columns="error_normalized",
    values="count"
).fillna(0)

pivot_stage.plot(kind="bar", figsize=(12, 6))
plt.ylabel("Count")
plt.title("Errors by pipeline stage")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

#### 3. Visualize most problematic files

In [ ]:
top_files = (
    df_results[df_results["status"] == "failed"]["file_name"]
    .value_counts()
    .head(10)
)

plt.figure(figsize=(10, 5))
plt.bar(top_files.index, top_files.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Failures")
plt.title("Most problematic workflow files")
plt.tight_layout()
plt.show()